<!-- SPDX-License-Identifier: Apache-2.0 -->
# UAIRT on macOS: build and test every Apple-silicon feature

This notebook builds UAIRT **once, with every backend that can use a Mac's CPU, GPU (Metal) or Neural Engine**, tests each of them, and ends
with a table of what passed, failed or was skipped. Run it on a Mac (Apple silicon is the tested case; Intel Macs build but have no Neural Engine).

| Build option | What it gives | Checked here |
|---|---|---|
| `UAIRT_BUILD_COREML` | the `coreml` backend (`.mlpackage`, `.mlmodelc`), compute units CPU / GPU / Neural Engine | all four compute-unit settings against PyTorch, the repo's CoreML tests |
| `UAIRT_BUILD_ONNXRUNTIME` | the `onnxruntime` backend, CPU and ONNX Runtime's **CoreML provider** (there is no Metal provider) | CPU, and the CoreML provider with three compute-unit settings and `cpu_fallback=0` |
| `UAIRT_BUILD_MNN` + `UAIRT_MNN_METAL` | the `mnn` backend, CPU and **Metal** | float32, half precision and the Metal device against PyTorch |
| `UAIRT_BUILD_NCNN` | the `ncnn` backend (CPU) | float32 and half precision |
| `UAIRT_BUILD_OPENVINO` | the `openvino` backend (CPU), from the pip package | float32 and half precision |
| `UAIRT_BUILD_LLAMACPP` | GGUF language models, **Metal** by default | text on Metal against the CPU, layers actually offloaded, the repo's llama.cpp tests |
| `UAIRT_BUILD_WHISPERCPP` | speech to text, **Metal** by default | transcript on Metal against the CPU, the repo's comparison with `whisper-cli` |
| core, bindings | C API, loader, sessions, `libuairt_llm`, Python binding | the whole `ctest` run |

ExecuTorch is built too when you set `EXECUTORCH_ROOT` to your own install (it is not something this notebook can fetch); otherwise it is skipped with that reason.
Not covered: the Neural Engine is reached only through CoreML (the other backends cannot use it), and QNN, TensorRT and TFLite have no macOS path here.

**What this notebook installs.** With `INSTALL_FROM_PIP = True` (the next cell) it pip-installs into the Python that runs it, only what is missing:
`numpy`, `onnx`, `onnxruntime`, `coremltools`, `openvino`, `pytest`, and the optional `MNN`, `pnnx`, `ncnn` (MNN's package replaces `protobuf`, so use a throw-away virtual
environment). **PyTorch and torchvision are not installed for you** (they are large); without them the model checks are skipped with that reason.
It also fetches the pinned `llama.cpp`, `whisper.cpp`, `ncnn` and `MNN` submodules, and two small models (Whisper `tiny.en`, 78 MB, and SmolLM2-135M Q8, 145 MB)
checked against published hashes. The first build compiles llama.cpp, whisper.cpp, MNN and NCNN and takes several minutes.

## 1. The machine

In [1]:
import importlib.util, os, platform, re, shutil, subprocess, sys, time
from pathlib import Path

if platform.system() != "Darwin":
    raise SystemExit("This notebook is for macOS. For an NVIDIA GPU use uairt_tensorrt_colab.ipynb; elsewhere use uairt_backends_tour.ipynb.")

def sh(*command, **kwargs):
    return subprocess.run(command, capture_output=True, text=True, **kwargs)

chip = sh("sysctl", "-n", "machdep.cpu.brand_string").stdout.strip()
print(f"{chip} | macOS {platform.mac_ver()[0]} | {platform.machine()} | python {sys.version.split()[0]} ({sys.executable})")
print("Xcode tools:", sh("xcode-select", "-p").stdout.strip() or "NOT FOUND (run: xcode-select --install)")
print("cmake:", (sh("cmake", "--version").stdout.splitlines() or ["NOT FOUND (brew install cmake)"])[0])
assert shutil.which("cmake") and shutil.which("clang"), "cmake and a C/C++ compiler (Xcode command line tools) are required"
APPLE_SILICON = platform.machine() == "arm64"

# Every check below adds a row here; the last cell prints the table.
RESULTS = []
class Skip(Exception):
    pass

def check(area, name):
    """Runs the function now. Its return value is the detail; Skip(reason) marks a skip; any other exception is a failure."""
    def run(fn):
        try:
            detail = fn()
            RESULTS.append((area, name, "PASS", detail or ""))
        except Skip as why:
            RESULTS.append((area, name, "SKIP", str(why)))
        except BaseException as error:
            RESULTS.append((area, name, "FAIL", f"{type(error).__name__}: {str(error)[:300]}"))
        print(f"{RESULTS[-1][2]:<5} {area} / {name}: {RESULTS[-1][3]}")
        return fn
    return run

Apple M5 | macOS 26.6.2 | arm64 | python 3.12.15 (python)
Xcode tools: /Applications/Xcode.app/Contents/Developer
cmake: cmake version 4.4.4


## 2. Python packages

Edit the flags to opt out. A package that fails to install is reported and its checks are skipped; it never stops the notebook.

In [2]:
INSTALL_FROM_PIP = True
WANTED = {"numpy": "numpy", "onnx": "onnx", "onnxruntime": "onnxruntime", "coremltools": "coremltools", "openvino": "openvino", "pytest": "pytest"}
OPTIONAL = {"MNN": "MNN", "pnnx": "pnnx", "ncnn": "ncnn"}     # exporters for the MNN and NCNN checks

def have(module):
    return importlib.util.find_spec(module) is not None

missing = [pkg for mod, pkg in {**WANTED, **OPTIONAL}.items() if not have(mod)]
if missing and INSTALL_FROM_PIP:
    for pkg in missing:                       # one at a time, so one failure does not block the others
        done = sh(sys.executable, "-m", "pip", "install", "-q", pkg)
        print(("installed " if done.returncode == 0 else "COULD NOT INSTALL ") + pkg, "" if done.returncode == 0 else done.stderr.strip()[-200:])
elif missing:
    print("missing (INSTALL_FROM_PIP is off):", missing)

HAVE = {mod: have(mod) for mod in [*WANTED, *OPTIONAL, "torch", "torchvision", "executorch"]}
print({k: v for k, v in HAVE.items()})
if not (HAVE["torch"] and HAVE["torchvision"]):
    print("PyTorch / torchvision not installed: the model checks (section 6) will be skipped. pip install torch torchvision to enable them.")

import numpy as np

{'numpy': True, 'onnx': True, 'onnxruntime': True, 'coremltools': True, 'openvino': True, 'pytest': True, 'MNN': True, 'pnnx': True, 'ncnn': True, 'torch': True, 'torchvision': True, 'executorch': True}


## 3. The repository, its submodules and two models

The notebook uses the checkout it sits in, or clones one next to itself. Models are fetched into `models/` beside the notebook (git ignores `*.bin` and `*.gguf`)
and checked against their published hashes; a model already there is used as is.

In [3]:
NOTEBOOK_DIR = Path.cwd()
def find_repo():
    for folder in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (folder / "CMakeLists.txt").exists() and (folder / "include/uairt/uairt.h").exists():
            return folder
    return None
REPO = Path(os.environ["UAIRT_REPO"]) if os.environ.get("UAIRT_REPO") else find_repo()
if REPO is None:
    REPO = NOTEBOOK_DIR / "unified-ai-runtime"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/vnk8071/unified-ai-runtime.git", str(REPO)], check=True)
SUBMODULES = ["third_party/llama.cpp", "third_party/whisper.cpp", "third_party/ncnn", "third_party/MNN"]
subprocess.run(["git", "-C", str(REPO), "submodule", "update", "--init", "--depth", "1", *SUBMODULES], check=True)
print("repository:", REPO, "|", sh("git", "-C", str(REPO), "log", "-1", "--format=%h %s").stdout.strip())

import hashlib, urllib.request
MODELS = NOTEBOOK_DIR / "models"; MODELS.mkdir(exist_ok=True)
def fetch(filename, url, algorithm, expected):
    target = MODELS / filename
    if not target.exists():
        part = MODELS / (filename + ".part")
        urllib.request.urlretrieve(url, part)
        digest = hashlib.new(algorithm)
        with open(part, "rb") as stream:
            for block in iter(lambda: stream.read(1 << 20), b""):
                digest.update(block)
        if digest.hexdigest() != expected:
            part.unlink()
            raise SystemExit(f"{filename}: {algorithm} {digest.hexdigest()} is not the expected {expected}")
        part.rename(target)
    return target

WHISPER_MODEL = fetch("ggml-tiny.en.bin", "https://huggingface.co/ggerganov/whisper.cpp/resolve/main/ggml-tiny.en.bin",
                      "sha1", "c78c86eb1a8faa21b369bcd33207cc90d64ae9df")
LLAMA_MODEL = fetch("SmolLM2-135M-Instruct-Q8_0.gguf",
                    "https://huggingface.co/bartowski/SmolLM2-135M-Instruct-GGUF/resolve/main/SmolLM2-135M-Instruct-Q8_0.gguf",
                    "sha256", "5a1395716f7913741cc51d98581b9b1228d80987a9f7d3664106742eb06bba83")
JFK = REPO / "third_party/whisper.cpp/samples/jfk.wav"
print(WHISPER_MODEL.name, LLAMA_MODEL.name, "jfk.wav:", JFK.exists())

repository: ~/code/unified-ai-runtime | bed8509 Add the ONNX Runtime CUDA and TensorRT provider cells to the Colab notebook
ggml-tiny.en.bin SmolLM2-135M-Instruct-Q8_0.gguf jfk.wav: True


## 4. Build UAIRT with every Apple backend

One configuration, shared libraries (the Python binding and the Python tests need them). llama.cpp and whisper.cpp use Metal on Apple platforms without any flag; MNN needs
`UAIRT_MNN_METAL`. The environment variables are the ones `CMakeLists.txt` reads to switch on the comparisons with the vendor tools (`llama-completion`, `whisper-cli`, `coremltools`),
so `ctest` below compares UAIRT with each of them, not only with itself. ONNX Runtime's header is fetched into `.ort/` when you have none (the plugin loads the library at run time).
The log is in `build.log` inside the build folder.

In [4]:
BUILD = REPO / "build-macos-notebook"
os.environ.update({"UAIRT_LLAMACPP_TEST_MODEL": str(LLAMA_MODEL), "UAIRT_WHISPERCPP_TEST_MODEL": str(WHISPER_MODEL),
                   "UAIRT_WHISPERCPP_TEST_AUDIO": str(JFK)})
options = [f"-DPython3_EXECUTABLE={sys.executable}",       # ctest's Python tests run in the same environment as this notebook
           "-DCMAKE_BUILD_TYPE=Release", "-DBUILD_SHARED_LIBS=ON", "-DUAIRT_BUILD_COREML=ON", "-DUAIRT_BUILD_ONNXRUNTIME=ON",
           "-DUAIRT_BUILD_MNN=ON", "-DUAIRT_MNN_METAL=ON", "-DUAIRT_BUILD_NCNN=ON", "-DUAIRT_BUILD_LLAMACPP=ON", "-DUAIRT_BUILD_WHISPERCPP=ON"]
if HAVE["coremltools"]:
    os.environ["UAIRT_COREML_PYTHON"] = sys.executable
if HAVE["openvino"]:
    import openvino
    options += ["-DUAIRT_BUILD_OPENVINO=ON", f"-DOPENVINO_ROOT={Path(openvino.__file__).parent}"]
BUILT_EXECUTORCH = bool(os.environ.get("EXECUTORCH_ROOT"))
if BUILT_EXECUTORCH:
    options.append("-DUAIRT_BUILD_EXECUTORCH=ON")
if not (REPO / ".ort/include/onnxruntime_c_api.h").exists():
    subprocess.run(["bash", str(REPO / "scripts/ci/fetch_onnxruntime_headers.sh")], check=True, capture_output=True)

BUILD.mkdir(exist_ok=True)
t0 = time.time()
with open(BUILD / "build.log", "w") as log:
    for step in (["cmake", "-S", str(REPO), "-B", str(BUILD), *options],
                 ["cmake", "--build", str(BUILD), "-j", str(os.cpu_count() or 4)]):
        done = subprocess.run(step, stdout=log, stderr=subprocess.STDOUT, text=True)
        if done.returncode:
            break
print("\n".join((BUILD / "build.log").read_text().splitlines()[-8:]))
print(f"build {'FAILED' if done.returncode else 'finished'} in {(time.time() - t0) / 60:.1f} min")
assert done.returncode == 0, f"see {BUILD / 'build.log'}"
PLUGINS = sorted(p.name.replace("libuairt_backend_", "").split(".")[0] for p in BUILD.glob("libuairt_backend_*"))
print("plugins:", PLUGINS)

[ 72%] Built target MNN
[ 72%] Built target uairt_backend_mnn
[ 78%] Built target llama-common
[ 78%] Built target llama-completion-impl
[ 78%] Built target llama-completion
[ 78%] Built target llm_generate
[100%] Built target ncnn
[100%] Built target uairt_backend_ncnn
build finished in 0.1 min
plugins: ['coreml', 'llamacpp', 'mnn', 'ncnn', 'onnxruntime', 'openvino', 'whispercpp']


## 5. The repo's own tests

`ctest` runs every test the configuration enabled: the core and loader, sessions and `libuairt_llm`, each backend's error paths, the comparisons with the vendor tools named above,
the Python tests and the symbol checks. The `onnxruntime` test then runs once more with `UAIRT_TEST_ORT_PROVIDER=CoreML`, which makes it load a model through ONNX Runtime's CoreML provider.

In [5]:
LIBRARY = next(BUILD.glob("libuairt.dylib"))
ORT_LIBRARY = None
if HAVE["onnxruntime"]:
    import onnxruntime as ort
    ORT_LIBRARY = next((Path(ort.__file__).parent / "capi").glob("libonnxruntime*.dylib"))
    os.environ["UAIRT_ONNXRUNTIME_LIBRARY"] = str(ORT_LIBRARY)
env = dict(os.environ, UAIRT_LIBRARY=str(LIBRARY), UAIRT_PLUGIN_PATH=str(BUILD), PYTHONPATH=str(REPO / "bindings/python"))

done = subprocess.run(["ctest", "--test-dir", str(BUILD), "--output-on-failure", "--timeout", "900"], capture_output=True, text=True, env=env)
print("\n".join(done.stdout.splitlines()[-25:]))
for number, name, outcome in re.findall(r"Test\s+#(\d+):\s+(\S+)\s+\.+\s*(\*{0,3}\w[\w ]*?)\s+[\d.]+ sec", done.stdout):
    ok = outcome.strip() == "Passed"
    if not ok and name == "references_scripts" and (REPO / "docs/superpowers").exists():
        # that test scans every markdown file under docs/, including untracked local ones; a clean checkout does not have this folder
        RESULTS.append(("ctest", name, "SKIP", "fails only because of untracked files under docs/superpowers on this machine"))
        continue
    RESULTS.append(("ctest", name, "PASS" if ok else "FAIL", "" if ok else outcome.strip()))
assert any(r[0] == "ctest" for r in RESULTS), "ctest printed no test lines"

if HAVE["onnxruntime"]:
    one = subprocess.run(["ctest", "--test-dir", str(BUILD), "-R", "^onnxruntime$", "--output-on-failure"], capture_output=True, text=True,
                         env=dict(env, UAIRT_TEST_ORT_PROVIDER="CoreML"))
    RESULTS.append(("ctest", "onnxruntime with UAIRT_TEST_ORT_PROVIDER=CoreML", "PASS" if one.returncode == 0 else "FAIL",
                    "" if one.returncode == 0 else "\n".join(one.stdout.splitlines()[-6:])[:300]))
    print(RESULTS[-1][:3])

----------------------------------------------------------------------
Traceback (most recent call last):
  File "~/code/unified-ai-runtime/tests/test_references.py", line 44, in test_local_link_check_and_repo_has_none_broken
    self.assertEqual(refs.main(["--offline"]), 0)
AssertionError: 1 != 0

----------------------------------------------------------------------
Ran 5 tests in 0.010s

FAILED (failures=1)
BROKEN     local link ../LICENSING.md
           in docs/superpowers/plans/2026-10-05-llamacpp-sessions.md
BROKEN     local link llamacpp.md
           in docs/superpowers/plans/2026-10-05-llamacpp-sessions.md
checked 0 urls in 49 files: 0 broken, 0 unverified, 0 redirected, 2 broken local links

      Start 24: example_cpp_quickstart
24/24 Test #24: example_cpp_quickstart ...........   Passed    0.00 sec

96% tests passed, 1 tests failed out of 24

Total Test time (real) =  18.63 sec

The following tests FAILED:
	 23 - references_scripts (Failed)
('ctest', 'onnxruntime with UAIR

## 6. One model through every backend: `scripts/bench_model.py`

ResNet-18 (random weights, so nothing is downloaded) is exported to ONNX, CoreML, OpenVINO, NCNN, MNN (and ExecuTorch when built), and every backend and option variant
runs it **in its own process** (the native libraries of different runtimes can deadlock in one). Each result is compared with PyTorch and timed with the core's own run statistics.

A row is `ok` when it matches PyTorch to float32 rounding, and `reduced` when it is close but computed in half precision (the GPU and the Neural Engine do that); a
reduced row is a pass here, **not** an exact one, and the table says which is which. `FAILED` means NaN, constant or far-off output. The variants include
ONNX Runtime's CoreML provider with `cpu_fallback=0`, so a provider that silently could not start would fail instead of passing as the CPU.

In [6]:
import json, tempfile
if not (HAVE["torch"] and HAVE["torchvision"] and HAVE["onnx"]):
    BENCH = None
    RESULTS.append(("models", "ResNet-18 through every backend", "SKIP", "needs torch, torchvision and onnx (pip install torch torchvision)"))
    print(RESULTS[-1])
else:
    result_file = Path(tempfile.mkdtemp()) / "bench.json"
    command = [sys.executable, str(REPO / "scripts/bench_model.py"), "torchvision:resnet18", "--shape", "1,3,224,224",
               "--runs", "50", "--json", str(result_file)]
    done = subprocess.run(command, capture_output=True, text=True, env=env)
    print("\n".join(l for l in done.stdout.splitlines() if not l.startswith("[")))
    if not result_file.exists():
        print(done.stderr[-1500:])
    BENCH = json.loads(result_file.read_text()) if result_file.exists() else None
    assert BENCH, "the benchmark wrote no result"

    for r in BENCH["results"]:
        name = r["backend"] + (" " + ",".join(f"{k}={v}" for k, v in r["options"].items() if k != "input_shapes") if r["options"] else "")
        status = r.get("status")
        if status == "ok" and r["accuracy"] in ("ok", "reduced"):
            RESULTS.append(("models", name, "PASS", f"{r['median_ms']:.2f} ms, max diff {r['max_diff']:.1e} ({r['accuracy']})"))
        elif status == "ok":
            RESULTS.append(("models", name, "FAIL", f"{r['accuracy']}: max diff {r['max_diff']:.1e}"))
        elif status in ("skipped", "unavailable"):
            RESULTS.append(("models", name, "SKIP", r.get("error", status)[:160]))
        else:
            RESULTS.append(("models", name, "FAIL", r.get("error", status)[:200]))
        print(f"{RESULTS[-1][2]:<5} models / {name}: {RESULTS[-1][3]}")


Model: torchvision:resnet18, inputs [(1, 3, 224, 224)], 50 runs after 3 warm-up runs per variant
Machine: Apple M5 (macOS-26.6.2-arm64-arm-64bit); UAIRT 0.2.0

| Backend and options | Load ms | Median ms | p90 ms | Min ms | Max diff | Accuracy |
|---|---:|---:|---:|---:|---:|---|
| onnxruntime execution_provider=CoreML,ep_option.MLComputeUnits=CPUAndNeuralEngine,cpu_fallback=0 | 355.7 | 0.584 | 0.609 | 0.560 | 0.0135 | reduced |
| onnxruntime execution_provider=CoreML,ep_option.MLComputeUnits=ALL,cpu_fallback=0 | 364.9 | 0.585 | 0.600 | 0.556 | 0.0135 | reduced |
| mnn device=metal | 453.3 | 1.378 | 1.413 | 1.340 | 0.00412 | reduced |
| coreml compute_units=cpu_and_gpu | 136.3 | 1.502 | 1.554 | 1.474 | 2.03e-06 | ok |
| coreml compute_units=all | 142.3 | 1.506 | 1.531 | 1.468 | 2.03e-06 | ok |
| onnxruntime execution_provider=CoreML,ep_option.MLComputeUnits=CPUAndGPU,cpu_fallback=0 | 207.7 | 1.691 | 1.727 | 1.668 | 0.0242 | reduced |
| mnn precision=low | 133.8 | 3.130 | 3.444 | 3.066

## 7. llama.cpp on Metal

The same prompt, greedy, once with `n_gpu_layers=0` (CPU) and once with the default (all layers offloaded to Metal), each in its own process. Two checks beyond "it printed text":
**prompt processing** of a ~1,700-token prompt must be clearly faster on Metal (it is limited by arithmetic, which a GPU does far better than a 135M model's token-by-token
generation, so this separates the two where a short answer would not; llama.cpp's own log is silenced by the plugin, so speed is the evidence), and the GPU text must equal the CPU
text. Greedy output on a GPU can differ from the CPU's after some tokens because the floating-point sums are ordered differently; that is recorded, and only an empty or very
different answer fails.

In [7]:
LLAMA_SNIPPET = """
import sys, time, json, uairt
options = json.loads(sys.argv[2])
long_prompt = "The quick brown fox jumps over the lazy dog. " * 170
with uairt.Engine("llamacpp", options) as engine, engine.load_model(sys.argv[1]) as model:
    model.complete("Write one sentence about the ocean.", max_tokens=4, temperature=0)
    start = time.perf_counter()
    text = model.complete("Write one sentence about the ocean.", max_tokens=48, temperature=0, seed=1)
    generate_ms = (time.perf_counter() - start) * 1e3
    best = 1e9
    for _ in range(3):
        start = time.perf_counter()
        model.complete(long_prompt, max_tokens=1, temperature=0)
        best = min(best, (time.perf_counter() - start) * 1e3)
    print(json.dumps({"text": text, "ms": generate_ms, "prompt_ms": best}))
"""
def run_snippet(snippet, *args):
    done = subprocess.run([sys.executable, "-c", snippet, *args], capture_output=True, text=True, env=env)
    lines = [l for l in done.stdout.splitlines() if l.startswith("{")]
    if done.returncode or not lines:
        raise RuntimeError((done.stderr or done.stdout)[-400:])
    return json.loads(lines[-1]), done.stderr

LLAMA = {}
@check("llamacpp", "CPU generation (n_gpu_layers=0)")
def _():
    out, log = run_snippet(LLAMA_SNIPPET, str(LLAMA_MODEL), json.dumps({"n_gpu_layers": "0"}))
    LLAMA["cpu"] = out
    assert out["text"].strip(), "empty answer"
    return f"{out['ms']:.0f} ms for 48 tokens; ~1700-token prompt {out['prompt_ms']:.0f} ms; {out['text'].strip()[:50]!r}"

@check("llamacpp", "Metal generation (default device)")
def _():
    out, log = run_snippet(LLAMA_SNIPPET, str(LLAMA_MODEL), json.dumps({}))
    LLAMA["gpu"] = out
    assert out["text"].strip(), "empty answer"
    return f"{out['ms']:.0f} ms for 48 tokens; ~1700-token prompt {out['prompt_ms']:.0f} ms"

@check("llamacpp", "Metal processes a long prompt much faster than the CPU")
def _():
    if "gpu" not in LLAMA or "cpu" not in LLAMA:
        raise Skip("needs the two runs above")
    ratio = LLAMA["cpu"]["prompt_ms"] / LLAMA["gpu"]["prompt_ms"]
    assert ratio > 2.5, f"only {ratio:.1f}x faster: the model may not be on the GPU"
    return f"{ratio:.1f}x faster ({LLAMA['cpu']['prompt_ms']:.0f} ms against {LLAMA['gpu']['prompt_ms']:.0f} ms)"

@check("llamacpp", "the Metal device can be named, an unknown one is refused")
def _():
    named = run_snippet(LLAMA_SNIPPET, str(LLAMA_MODEL), json.dumps({"device": "MTL0"}))[0]
    assert named["text"].strip()
    try:
        run_snippet(LLAMA_SNIPPET, str(LLAMA_MODEL), json.dumps({"device": "NoSuchDevice"}))
    except RuntimeError as error:
        return f"device=MTL0 works ({named['prompt_ms']:.0f} ms prompt); an unknown name was refused"
    raise AssertionError("an unknown device was accepted")

@check("llamacpp", "Metal text equals CPU text")
def _():
    if "gpu" not in LLAMA or "cpu" not in LLAMA:
        raise Skip("needs the two runs above")
    cpu, gpu = LLAMA["cpu"]["text"], LLAMA["gpu"]["text"]
    if cpu == gpu:
        return f"identical ({len(cpu)} characters)"
    first = next((i for i, (a, b) in enumerate(zip(cpu, gpu)) if a != b), min(len(cpu), len(gpu)))
    assert first > 8, f"the texts differ from character {first}: {cpu[:60]!r} vs {gpu[:60]!r}"
    return f"equal for {first} characters, then differ (floating-point order)"

PASS  llamacpp / CPU generation (n_gpu_layers=0): 113 ms for 48 tokens; ~1700-token prompt 775 ms; 'The ocean is a vast, mysterious, and awe-inspiring'


PASS  llamacpp / Metal generation (default device): 97 ms for 48 tokens; ~1700-token prompt 99 ms
PASS  llamacpp / Metal processes a long prompt much faster than the CPU: 7.8x faster (775 ms against 99 ms)


PASS  llamacpp / the Metal device can be named, an unknown one is refused: device=MTL0 works (100 ms prompt); an unknown name was refused
PASS  llamacpp / Metal text equals CPU text: equal for 116 characters, then differ (floating-point order)


## 8. whisper.cpp on Metal

`jfk.wav` (11 s, 16 kHz) transcribed with `use_gpu=false` and with the default. Again one process each, reading whisper.cpp's own log for the backend it picked. The transcripts must agree
after ignoring case, punctuation and spacing.

In [8]:
WHISPER_SNIPPET = """
import sys, time, json, wave, numpy as np, uairt
with wave.open(sys.argv[2], "rb") as audio:
    pcm = np.frombuffer(audio.readframes(audio.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
with uairt.Engine("whispercpp", n_samples=len(pcm), use_gpu=sys.argv[3]) as engine, engine.load_model(sys.argv[1]) as model:
    model.run(pcm)
    start = time.perf_counter()
    text, length = model.run(pcm)
    print(json.dumps({"text": bytes(text[: int(length[0])]).decode("utf-8").strip(), "ms": (time.perf_counter() - start) * 1e3}))
"""
def normal(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

WHISPER = {}
@check("whispercpp", "CPU transcript (use_gpu=false)")
def _():
    out, log = run_snippet(WHISPER_SNIPPET, str(WHISPER_MODEL), str(JFK), "false")
    WHISPER["cpu"] = out["text"]
    assert "ask not" in normal(out["text"]), out["text"]
    assert re.search(r"use gpu\s*=\s*0", log), "whisper.cpp did not report use gpu = 0"
    assert not re.search(r"whisper_backend_init_gpu: using \w+ backend", log), "whisper.cpp picked a GPU backend although use_gpu=false"
    return f"{out['ms']:.0f} ms; use gpu = 0 (its log: no GPU found); {out['text'][:60]!r}"

@check("whispercpp", "Metal transcript (use_gpu=true) uses the GPU backend")
def _():
    out, log = run_snippet(WHISPER_SNIPPET, str(WHISPER_MODEL), str(JFK), "true")
    WHISPER["gpu"] = out["text"]
    picked = re.search(r"whisper_backend_init_gpu: using (\w+) backend", log)
    assert re.search(r"use gpu\s*=\s*1", log) and picked, "whisper.cpp's log shows no GPU backend, so the GPU use is not proven"
    assert "metal" in log.lower(), f"whisper.cpp picked the {picked.group(1)} backend, but its log never mentions Metal"
    return f"{out['ms']:.0f} ms; use gpu = 1; backend {picked.group(1)} (Metal)"

@check("whispercpp", "Metal transcript equals CPU transcript")
def _():
    if "gpu" not in WHISPER or "cpu" not in WHISPER:
        raise Skip("needs the two runs above")
    assert normal(WHISPER["cpu"]) == normal(WHISPER["gpu"]), (WHISPER["cpu"], WHISPER["gpu"])
    return "same words"

PASS  whispercpp / CPU transcript (use_gpu=false): 111 ms; use gpu = 0 (its log: no GPU found); 'And so my fellow Americans ask not what your country can do '


PASS  whispercpp / Metal transcript (use_gpu=true) uses the GPU backend: 45 ms; use gpu = 1; backend MTL0 (Metal)
PASS  whispercpp / Metal transcript equals CPU transcript: same words


## 9. Summary

In [9]:
width = max(len(a) + len(n) for a, n, _, _ in RESULTS) + 5
for area, name, status, detail in RESULTS:
    if area != "ctest" or status != "PASS":
        print(f"{status:<5} {area} / {name}".ljust(width + 6), detail[:140])
passed = sum(r[2] == "PASS" for r in RESULTS); failed = sum(r[2] == "FAIL" for r in RESULTS); skipped = sum(r[2] == "SKIP" for r in RESULTS)
ctests = [r for r in RESULTS if r[0] == "ctest"]
print(f"\nctest: {sum(r[2] == 'PASS' for r in ctests)} of {len(ctests)} passed (the passing ones are not listed above)")
print(f"all checks: {passed} passed, {failed} failed, {skipped} skipped")
if not BUILT_EXECUTORCH:
    print("ExecuTorch was not built (set EXECUTORCH_ROOT to your install and rerun from the build cell).")
assert not failed, "some checks failed; the rows above say which"

SKIP  ctest / references_scripts                                                                                  fails only because of untracked files under docs/superpowers on this machine
PASS  models / coreml compute_units=cpu_only                                                                      3.92 ms, max diff 4.3e-06 (ok)
PASS  models / coreml compute_units=cpu_and_gpu                                                                   1.50 ms, max diff 2.0e-06 (ok)
PASS  models / coreml compute_units=cpu_and_ne                                                                    3.88 ms, max diff 4.3e-06 (ok)
PASS  models / coreml compute_units=all                                                                           1.51 ms, max diff 2.0e-06 (ok)
SKIP  models / executorch                                                                                         plugin not found (UAIRT_PLUGIN_PATH)
PASS  models / mnn precision=high                                             

## Troubleshooting

- **The build fails in `third_party/MNN` or `ncnn`:** run `git submodule update --init --depth 1 third_party/MNN third_party/ncnn` (cell 3 does), then read `build-macos-notebook/build.log`.
- **`references_scripts` shows as skipped:** that test scans every markdown file under `docs/`, including untracked ones such as a local `docs/superpowers/`; a clean checkout passes it.
- **A stale `uairt` installed with pip (editable or not) is imported instead of this checkout:** `pip uninstall uairt`, then restart the kernel. The notebook sets `PYTHONPATH` for the programs it starts.
- **The `models` rows say `skipped: plugin not found`:** that backend was not built. The row names it; see the `plugins:` line printed by the build cell.
- **A row says `unavailable`:** the device is not there (for example the Neural Engine on an Intel Mac). It did not run on the CPU instead.
- **ONNX Runtime and OpenVINO or PyTorch hang when used in one process:** their libraries conflict. The notebook always runs them in separate processes; do the same in your own code.